# Scrapline — TRELLIS on Colab's free T4 (045)
Concepts in, machine-part models out, at about a minute a model (the free Hugging Face allowance makes one or two a day).

**Once per new runtime:**
1. **Runtime → Change runtime type → T4 GPU, Runtime version 2025.07.** That runtime is Python 3.11 with torch 2.6 / CUDA 12.4,
   which every TRELLIS dependency has a build for. The latest runtime is Python 3.13, which spconv has no wheel for.
2. Run cell 1 (install, ~20 min: it builds nvdiffrast and the Gaussian rasterizer for the T4), then **Runtime → Restart session**
   (the install replaces packages the session already loaded: Pillow, numpy).

**Each batch:** upload the cut concepts (`tools/gen3d/colab_prep.sh <ids>` → `shots/colab_in/*.png`) with the Files panel into
`/content`, run cell 2, download `scrapline_parts.zip`, then `tools/gen3d/colab_prep.sh --import <unzipped folder>`.
Settings are the Hugging Face demo's (sparse 12 steps / cfg 7.5, latent 12 / cfg 3.0, simplify 0.95, 1024 texture, seed 0);
a model already in `/content/out` is skipped.

In [ ]:
# 1. Install (once per runtime; ~20 min). Keeps the runtime's torch 2.6.0+cu124.
%cd /content
!test -d TRELLIS || git clone -q --recurse-submodules https://github.com/microsoft/TRELLIS.git
!pip -q install xformers==0.0.29.post3 --index-url https://download.pytorch.org/whl/cu124
!pip -q install spconv-cu120 easydict rembg onnxruntime trimesh xatlas igraph imageio imageio-ffmpeg opencv-python-headless
# The newest pyvista needs a newer IPython than this runtime has; 0.43 needs the vtk of its day.
!pip -q install 'pyvista==0.43.10' 'vtk==9.3.1' pymeshfix
!pip -q install git+https://github.com/EasternJournalist/utils3d.git@9a4eb15e4021b67b12c460c7057d642626897ec8
import os
os.environ['TORCH_CUDA_ARCH_LIST'] = '7.5'   # the T4: builds only what it runs
!test -d /tmp/ext/nvdiffrast || git clone -q https://github.com/NVlabs/nvdiffrast.git /tmp/ext/nvdiffrast
!pip -q install --no-build-isolation /tmp/ext/nvdiffrast
!test -d /tmp/ext/mip-splatting || git clone -q https://github.com/autonomousvision/mip-splatting.git /tmp/ext/mip-splatting
!pip -q install --no-build-isolation /tmp/ext/mip-splatting/submodules/diff-gaussian-rasterization/
print('installed: now Runtime -> Restart session, then run cell 2')

In [ ]:
# 2. Every /content/*.png without a model in /content/out, then a zip of them all.
import os, sys, glob, time, gc, shutil, types
os.environ['ATTN_BACKEND'] = 'xformers'      # the T4 cannot run flash-attn 2
os.environ['SPCONV_ALGO'] = 'native'
os.environ['CUMM_DISABLE_JIT'] = '1'         # cumm thinks it is an editable install and tries to rebuild itself
os.environ['SPCONV_DISABLE_JIT'] = '1'
sys.path.insert(0, '/content/TRELLIS')
# Imported by code paths this notebook never takes: text-to-3D (open3d) and flexicubes' shape asserts (kaolin).
o3d = types.ModuleType('open3d'); o3d.geometry = types.SimpleNamespace(TriangleMesh=object)
sys.modules['open3d'] = o3d
for name in ('kaolin', 'kaolin.utils', 'kaolin.utils.testing'):
    sys.modules[name] = types.ModuleType(name)
sys.modules['kaolin.utils.testing'].check_tensor = lambda *a, **k: True
import torch
from PIL import Image
from trellis.pipelines import TrellisImageTo3DPipeline
from trellis.utils import postprocessing_utils
if 'pipeline' not in globals():
    pipeline = TrellisImageTo3DPipeline.from_pretrained('JeffreyXiang/TRELLIS-image-large')
    pipeline.cuda()
os.makedirs('/content/out', exist_ok=True)
print('inputs:', sorted(glob.glob('/content/*.png')))
for path in sorted(glob.glob('/content/*.png')):
    part = os.path.splitext(os.path.basename(path))[0]
    target = '/content/out/%s.glb' % part
    if os.path.exists(target):
        continue
    t = time.time()
    out = pipeline.run(Image.open(path), seed=0,
                       sparse_structure_sampler_params={'steps': 12, 'cfg_strength': 7.5},
                       slat_sampler_params={'steps': 12, 'cfg_strength': 3.0},
                       formats=['gaussian', 'mesh'])
    glb = postprocessing_utils.to_glb(out['gaussian'][0], out['mesh'][0], simplify=0.95, texture_size=1024, verbose=False)
    glb.export(target)
    print('%s: %.0f s, %.1f MB' % (part, time.time() - t, os.path.getsize(target) / 1e6))
    del out, glb; gc.collect(); torch.cuda.empty_cache()
shutil.make_archive('/content/scrapline_parts', 'zip', '/content/out')
print(sorted(os.listdir('/content/out')))